# Phase 2 — Agent DNA & Reasoning Operator Library

This notebook is the end-to-end demonstration for **Phase 2** of the
Evolvable Mathematical Reasoning Engine (MRE).

| Section | What you'll see |
|---------|----------------|
| §1 | AgentDNA construction, serialisation, mutation, crossover |
| §2 | Each operator in isolation (unit demos) |
| §3 | Full pipeline on real algebra / logic problems |
| §4 | TaskManager — multi-agent pool, scoring, fitness |
| §5 | OperatorStats leaderboard & macro-operator discovery |
| §6 | Evolution preview: mutant vs parent performance |


In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))

from mre.agents.dna import AgentDNA, PROMPT_TEMPLATES, KNOWN_DOMAINS
from mre.agents.state import ReasoningState
from mre.agents.agent import ReasoningAgent, ReasoningAgentPool
from mre.agents.task_manager import TaskManager
from mre.operators import (
    SymbolicSimplify, DeductiveStep, ProofByContradiction,
    EquationSolve, SelfCritique, RepairChain,
    OperatorPipeline, list_operators,
)
from mre.operators.stats import OperatorStats

print('✓ All Phase-2 modules imported')
print('Registered operators:', list(list_operators().keys()))

✓ All Phase-2 modules imported
Registered operators: ['SymbolicSimplify', 'DeductiveStep', 'ProofByContradiction', 'EquationSolve', 'SelfCritique', 'RepairChain']


## §1 — Agent DNA


In [2]:
# ── 1a. Default DNA ────────────────────────────────────────────────────────
dna = AgentDNA()
print(dna.pretty())

╔══ AgentDNA ══════════════════════════════════════╗
  ID          : 1bef2ce1  (hash 8e461e03)
  Generation  : 0
  Parents     : seed
  Fitness     : None
  ── Genes ──────────────────────────────────────
  model_gene  : claude-sonnet-4-20250514
  prompt_gene : 'rigorous'
  domain_gene : algebra (top)
  tool_gene   : ['sympy', 'python_exec', 'theorem_db', 'numerical_solver', 'latex_renderer']
  reason_gene : SymbolicSimplify → DeductiveStep → EquationSolve → SelfCritique → RepairChain
╚══════════════════════════════════════════════════╝


In [3]:
# ── 1b. Specialist DNA: algebra expert ────────────────────────────────────
import random

algebra_dna = AgentDNA(
    model_gene='claude-sonnet-4-20250514',
    prompt_gene='rigorous',
    domain_gene={d: (0.7 if d == 'algebra' else 0.3 / (len(KNOWN_DOMAINS) - 1))
                 for d in KNOWN_DOMAINS},
    tool_gene={'sympy': 0.9, 'python_exec': 0.5, 'theorem_db': 0.3,
               'numerical_solver': 0.4, 'latex_renderer': 0.2},
    reasoning_gene=['SymbolicSimplify', 'EquationSolve', 'SelfCritique'],
)
print(algebra_dna.pretty())

╔══ AgentDNA ══════════════════════════════════════╗
  ID          : 3c887331  (hash 7777340c)
  Generation  : 0
  Parents     : seed
  Fitness     : None
  ── Genes ──────────────────────────────────────
  model_gene  : claude-sonnet-4-20250514
  prompt_gene : 'rigorous'
  domain_gene : algebra (top)
  tool_gene   : ['sympy', 'python_exec', 'numerical_solver']
  reason_gene : SymbolicSimplify → EquationSolve → SelfCritique
╚══════════════════════════════════════════════════╝


In [4]:
# ── 1c. Serialise / deserialise ────────────────────────────────────────────
json_str = algebra_dna.to_json()
restored = AgentDNA.from_json(json_str)
assert restored.dna_hash() == algebra_dna.dna_hash(), 'Hash mismatch!'
print('✓ Serialise → JSON → deserialise round-trip OK')
print(f'  DNA hash: {algebra_dna.dna_hash()}')

✓ Serialise → JSON → deserialise round-trip OK
  DNA hash: 7777340c


In [5]:
# ── 1d. Mutation ───────────────────────────────────────────────────────────
rng = random.Random(42)
mutant = algebra_dna.mutate(rng=rng, mutation_rate=0.8)
print('Parent reasoning_gene :', algebra_dna.reasoning_gene)
print('Mutant reasoning_gene :', mutant.reasoning_gene)
print('Parent prompt          :', algebra_dna.prompt_gene)
print('Mutant prompt          :', mutant.prompt_gene)
domain_shift = {d: round(mutant.domain_gene[d] - algebra_dna.domain_gene[d], 4)
                for d in KNOWN_DOMAINS if abs(mutant.domain_gene[d] - algebra_dna.domain_gene[d]) > 1e-6}
print('Domain shifts          :', domain_shift)

Parent reasoning_gene : ['SymbolicSimplify', 'EquationSolve', 'SelfCritique']
Mutant reasoning_gene : ['SymbolicSimplify', 'EquationSolve', 'SelfCritique']
Parent prompt          : rigorous
Mutant prompt          : rigorous
Domain shifts          : {'algebra': -0.0186, 'number_theory': -0.0009, 'geometry': -0.0009, 'calculus': 0.0257, 'combinatorics': -0.0009, 'probability': -0.0009, 'linear_algebra': -0.0009, 'topology': -0.0009, 'logic': -0.0009, 'set_theory': -0.0009}


In [6]:
# ── 1e. Crossover ─────────────────────────────────────────────────────────
geometry_dna = AgentDNA(
    prompt_gene='creative',
    domain_gene={d: (0.8 if d == 'geometry' else 0.2 / (len(KNOWN_DOMAINS) - 1))
                 for d in KNOWN_DOMAINS},
    reasoning_gene=['DeductiveStep', 'ProofByContradiction', 'SelfCritique', 'RepairChain'],
)

child_a, child_b = AgentDNA.crossover(algebra_dna, geometry_dna, rng=rng)
print('Parent A ops :', algebra_dna.reasoning_gene)
print('Parent B ops :', geometry_dna.reasoning_gene)
print('Child  A ops :', child_a.reasoning_gene)
print('Child  B ops :', child_b.reasoning_gene)
print(f'Child A top domain: {child_a.top_domain}')
print(f'Child B top domain: {child_b.top_domain}')

Parent A ops : ['SymbolicSimplify', 'EquationSolve', 'SelfCritique']
Parent B ops : ['DeductiveStep', 'ProofByContradiction', 'SelfCritique', 'RepairChain']
Child  A ops : ['SymbolicSimplify', 'EquationSolve', 'SelfCritique']
Child  B ops : ['DeductiveStep', 'ProofByContradiction', 'SelfCritique', 'RepairChain', 'SelfCritique']
Child A top domain: geometry
Child B top domain: geometry


## §2 — Operator Isolation Demos


In [7]:
# ── 2a. SymbolicSimplify ───────────────────────────────────────────────────
state = ReasoningState.from_problem('simplify (x**2 - 1)/(x - 1)')
state = state.evolve(current_expression='(x**2 - 1)/(x - 1)')
result = SymbolicSimplify().apply(state)
print(f'Input  : {state.current_expression}')
print(f'Output : {result.current_expression}')
print(f'Method : {result.context.get("simplify_method")}')

Input  : (x**2 - 1)/(x - 1)
Output : x + 1
Method : simplify


In [8]:
# ── 2b. EquationSolve ─────────────────────────────────────────────────────
state = ReasoningState.from_problem('Solve x**2 - 5*x + 6 = 0')
state = state.evolve(context={'equation': 'x**2 - 5*x + 6 = 0'})
result = EquationSolve().apply(state)
print(f'Equation  : {state.context["equation"]}')
print(f'Solutions : {result.context["solutions"]}')
print(f'Is solved : {result.is_solved}')

Equation  : x**2 - 5*x + 6 = 0
Solutions : [2, 3]
Is solved : True


In [9]:
# ── 2c. DeductiveStep ─────────────────────────────────────────────────────
rules = [
    {'if': 'n is prime', 'then': 'n has exactly two divisors'},
    {'if': 'n is prime', 'then': 'n > 1'},
    {'if': 'n has exactly two divisors', 'then': 'n is not composite'},
]
state = ReasoningState.from_problem('What can we say about a prime number?')
state = state.evolve(
    current_expression='n is prime',
    context={'rules': rules},
)
result = DeductiveStep().apply(state)
print(f'Initial     : {state.current_expression}')
print(f'First deduction : {result.current_expression}')
print(f'All deduced : {result.context["deduced"]}')

Initial     : n is prime
First deduction : n has exactly two divisors
All deduced : ['n has exactly two divisors', 'n > 1']


In [10]:
# ── 2d. ProofByContradiction ──────────────────────────────────────────────
state = ReasoningState.from_problem('Prove: if x=1 and x=2 simultaneously, contradiction')
state = state.evolve(
    current_expression='x = 1',
    context={'assumptions': ['x - 1', 'x - 2']},
)
result = ProofByContradiction().apply(state)
print(f'Contradiction found : {result.context.get("contradiction_found")}')
print(f'Method              : {result.context.get("contradiction_method")}')
print(f'Current expression  : {result.current_expression}')

Contradiction found : True
Method              : system_inconsistency
Current expression  : Proved by contradiction: assumptions are inconsistent.


In [11]:
# ── 2e. SelfCritique (heuristic mode) ─────────────────────────────────────
from mre.agents.state import StepRecord

state = ReasoningState.from_problem('Solve x = 1')
# Inject a circular step
r1 = StepRecord('SymbolicSimplify', 'x', 'x + 0', True)
r2 = StepRecord('EquationSolve', 'x + 0', 'x + 0', True)  # same output → circular
state = state.evolve(history=[r1, r2], current_expression='x + 0')

result = SelfCritique(llm_client=None).apply(state)
critique = result.context.get('critique', {})
print('Issues     :', critique.get('issues'))
print('Suggestions:', critique.get('suggestions'))
print('Confidence :', result.confidence, '(was 1.0)')

Issues     : ["Circular step detected: step 1 and step 2 both produce 'x + 0'"]
Suggestions: ['Free symbols remain: {x} — consider running EquationSolve.']
Confidence : 0.7 (was 1.0)


In [12]:
# ── 2f. RepairChain ────────────────────────────────────────────────────────
state = ReasoningState.from_problem('Solve 2*x + 4 = 0')
r_fail = StepRecord('EquationSolve', 'bad input', '', success=False,
                    error_msg='Could not parse equation: bad input')
state = state.evolve(history=[r_fail], failed=True,
                     failure_reason='parse error', current_expression='??')

result = RepairChain().apply(state)
print(f'Before repair: failed={state.failed}, expr={state.current_expression!r}')
print(f'After repair : failed={result.failed}, expr={result.current_expression!r}')
print(f'Action taken : {result.context.get("last_repair_action")}')

Before repair: failed=True, expr='??'
After repair : failed=False, expr='Solve 2*x + 4 = 0'
Action taken : reset_to_problem


## §3 — Full Pipeline on Real Problems


In [13]:
# ── 3a. Algebra problem ────────────────────────────────────────────────────
problems = [
    {'text': 'Solve 3*x**2 - 12 = 0',
     'context': {'equation': '3*x**2 - 12 = 0'},
     'expected': '[-2, 2]'},
    {'text': 'Simplify sin(x)**2 + cos(x)**2',
     'context': {},
     'expected': '1'},
    {'text': 'Solve x + 7 = 15',
     'context': {'equation': 'x + 7 = 15'},
     'expected': '8'},
]

pipeline = OperatorPipeline.from_names(
    ['SymbolicSimplify', 'EquationSolve', 'SelfCritique', 'RepairChain']
)

for p in problems:
    state = ReasoningState.from_problem(p['text'])
    state = state.evolve(context={**state.context, **p['context']})
    res = pipeline.run(state)
    correct = p['expected'] in str(res.answer or '')
    marker = '✓' if correct else '✗'
    print(f'{marker} {p["text"][:50]:50s} | answer={res.answer}')

✓ Solve 3*x**2 - 12 = 0                              | answer=Solution for [x]: [-2, 2]
✓ Simplify sin(x)**2 + cos(x)**2                     | answer=1
✓ Solve x + 7 = 15                                   | answer=Solution for [x]: [8]


In [14]:
# ── 3b. Detailed pipeline report ──────────────────────────────────────────
state = ReasoningState.from_problem('Solve x**3 - 6*x**2 + 11*x - 6 = 0')
state = state.evolve(context={'equation': 'x**3 - 6*x**2 + 11*x - 6 = 0'})
result = pipeline.run(state)
print(result.report())

╔══ Pipeline Result ══════════════════════════════════╗
  Solved   : True
  Answer   : Solution for [x]: [1, 2, 3]
  Duration : 0.019s
  Operators: SymbolicSimplify → EquationSolve → SelfCritique → RepairChain
  ── Steps ──────────────────────────────────────────
    SymbolicSimplify          0.0069s
    EquationSolve             0.0125s
╚══════════════════════════════════════════════════════╝


## §4 — TaskManager: Multi-Agent Pool


In [15]:
# ── 4a. Build a diverse population ────────────────────────────────────────
population = [
    AgentDNA(reasoning_gene=['SymbolicSimplify', 'EquationSolve', 'SelfCritique'],
             prompt_gene='rigorous'),
    AgentDNA(reasoning_gene=['EquationSolve', 'SelfCritique', 'RepairChain'],
             prompt_gene='concise'),
    AgentDNA(reasoning_gene=['SymbolicSimplify', 'EquationSolve'],
             prompt_gene='creative'),
    AgentDNA(reasoning_gene=['DeductiveStep', 'EquationSolve', 'SelfCritique', 'RepairChain'],
             prompt_gene='adversarial'),
]

manager = TaskManager(
    population=population,
    max_rounds=2,
    target_score=0.80,
)

report = manager.run(
    'Solve x**2 - 5*x + 6 = 0',
    context={'equation': 'x**2 - 5*x + 6 = 0'},
    expected_answer='2',
)
print(report.summary())

21:40:07  INFO      mre.agents.task_manager  Round 1/2 — 4 agents
21:40:07  INFO      mre.agents.agent  Agent 9f7c3211 solving: Solve x**2 - 5*x + 6 = 0
21:40:07  INFO      mre.agents.agent  Agent 9f7c3211 finished: solved=True, answer=Solution for [x]: [2, 3]
21:40:07  INFO      mre.agents.agent  Agent 7d295264 solving: Solve x**2 - 5*x + 6 = 0
21:40:07  INFO      mre.agents.agent  Agent 7d295264 finished: solved=True, answer=Solution for [x]: [2, 3]
21:40:07  INFO      mre.agents.agent  Agent a3bc6763 solving: Solve x**2 - 5*x + 6 = 0
21:40:07  INFO      mre.agents.agent  Agent a3bc6763 finished: solved=True, answer=Solution for [x]: [2, 3]
21:40:07  INFO      mre.agents.agent  Agent b2d00678 solving: Solve x**2 - 5*x + 6 = 0
21:40:07  INFO      mre.agents.agent  Agent b2d00678 finished: solved=True, answer=Solution for [x]: [2, 3]
21:40:07  INFO      mre.agents.task_manager  Round 1 best score: 0.9900
21:40:07  INFO      mre.agents.task_manager  Target score 0.80 reached — stopping 

In [16]:
# ── 4b. Fitness scores written back into DNA ───────────────────────────────
print('Population after task (sorted by fitness):')
for dna in manager.sorted_population():
    print(f'  {dna.agent_id}  fitness={dna.fitness_score:.4f}  '
          f'ops={dna.reasoning_gene}  prompt={dna.prompt_gene}')

Population after task (sorted by fitness):
  7d295264  fitness=0.9900  ops=['EquationSolve', 'SelfCritique', 'RepairChain']  prompt=concise
  9f7c3211  fitness=0.8900  ops=['SymbolicSimplify', 'EquationSolve', 'SelfCritique']  prompt=rigorous
  a3bc6763  fitness=0.8900  ops=['SymbolicSimplify', 'EquationSolve']  prompt=creative
  b2d00678  fitness=0.8900  ops=['DeductiveStep', 'EquationSolve', 'SelfCritique', 'RepairChain']  prompt=adversarial


## §5 — OperatorStats Leaderboard & Macro-Operator Discovery


In [17]:
# ── 5a. Run many problems to build stats ──────────────────────────────────
import random as _random

problems_batch = [
    {'text': f'Solve {a}*x + {b} = 0',
     'ctx': {'equation': f'{a}*x + {b} = 0'}}
    for a, b in [(2,4),(3,-9),(1,5),(-2,6),(4,-8),(5,10),(1,-7),(3,3)]
] + [
    {'text': f'Simplify x**2 - {k}', 'ctx': {}}
    for k in [1, 4, 9, 16]
]

stats = OperatorStats()
pipeline_full = OperatorPipeline.from_names(
    ['SymbolicSimplify', 'EquationSolve', 'SelfCritique', 'RepairChain']
)

for p in problems_batch:
    s = ReasoningState.from_problem(p['text'])
    s = s.evolve(context={**s.context, **p['ctx']})
    r = pipeline_full.run(s)
    stats.record(r)

print(stats.leaderboard())

╔══ Operator Leaderboard ════════════════════════════════════╗
  Total runs: 12  |  Solved: 12
  Operator                        Calls Success%  AvgDur(s)
  ──────────────────────────────────────────────────────────
  SymbolicSimplify                   12   100.0%     0.0109
  EquationSolve                       8   100.0%     0.0032
╚══════════════════════════════════════════════════════════╝


In [18]:
# ── 5b. Top operator sequences ─────────────────────────────────────────────
print('Top operator sequences by solve-rate × log(uses):')
for seq_stats in stats.top_sequences(min_uses=2, top_k=5):
    print(f'  {" → ".join(seq_stats.sequence):50s}  '
          f'uses={seq_stats.uses}  solve_rate={seq_stats.solve_rate:.2f}  '
          f'score={seq_stats.score():.3f}')

Top operator sequences by solve-rate × log(uses):
  SymbolicSimplify → EquationSolve                    uses=8  solve_rate=1.00  score=2.197


In [19]:
# ── 5c. Macro-operator candidates ─────────────────────────────────────────
macros = stats.suggest_macro_operators(min_solve_rate=0.5, min_uses=2)
if macros:
    print('Macro-operator candidates (high solve-rate sequences):')
    for m in macros:
        print(' ', ' → '.join(m))
else:
    print('(No macro candidates yet — run more problems to accumulate data.)')

print('\nStats JSON preview:')
import json
d = stats.to_dict()
d.pop('top_sequences', None)  # trim for display
print(json.dumps(d, indent=2)[:800])

Macro-operator candidates (high solve-rate sequences):
  SymbolicSimplify → EquationSolve

Stats JSON preview:
{
  "total_runs": 12,
  "solved_runs": 12,
  "operators": {
    "SymbolicSimplify": {
      "name": "SymbolicSimplify",
      "calls": 12,
      "successes": 12,
      "success_rate": 1.0,
      "avg_duration_sec": 0.01088,
      "avg_confidence_gain": 0.0
    },
    "EquationSolve": {
      "name": "EquationSolve",
      "calls": 8,
      "successes": 8,
      "success_rate": 1.0,
      "avg_duration_sec": 0.00315,
      "avg_confidence_gain": 0.0
    }
  }
}


## §6 — Evolution Preview: Mutant vs Parent


In [20]:
# ── 6a. Seed from best performer, produce mutant generation ───────────────
best_dna = manager.sorted_population()[0]
print('Best parent:', best_dna.reasoning_gene, 'fitness:', best_dna.fitness_score)

rng = _random.Random(7)
mutants = [best_dna.mutate(rng=rng, mutation_rate=0.5) for _ in range(3)]
for m in mutants:
    print(f'  Mutant {m.agent_id}: {m.reasoning_gene}  prompt={m.prompt_gene}')

Best parent: ['EquationSolve', 'SelfCritique', 'RepairChain'] fitness: 0.99
  Mutant b42e32a6: ['EquationSolve', 'SelfCritique', 'RepairChain']  prompt=creative
  Mutant 3e7a6c2b: ['EquationSolve', 'SelfCritique']  prompt=rigorous
  Mutant 66ffba87: ['SelfCritique', 'RepairChain']  prompt=concise


In [21]:
# ── 6b. Race parent vs mutants on 4 equations ─────────────────────────────
race_problems = [
    ('Solve x**2 - 4 = 0', {'equation': 'x**2 - 4 = 0'}, '[-2, 2]'),
    ('Solve 2*x - 10 = 0', {'equation': '2*x - 10 = 0'}, '5'),
    ('Solve x**2 + x - 6 = 0', {'equation': 'x**2 + x - 6 = 0'}, '[-3, 2]'),
    ('Solve x - 3 = 0', {'equation': 'x - 3 = 0'}, '3'),
]

from mre.agents.task_manager import _default_scorer

def evaluate_dna(dna, problems):
    agent = ReasoningAgent(dna=dna)
    scores = []
    for text, ctx, expected in problems:
        res = agent.solve(text, context=ctx)
        scores.append(_default_scorer(res, expected))
    return sum(scores) / len(scores)

print(f'Parent fitness  : {evaluate_dna(best_dna, race_problems):.4f}')
for i, m in enumerate(mutants, 1):
    print(f'Mutant {i} fitness: {evaluate_dna(m, race_problems):.4f}  '
          f'ops={m.reasoning_gene}')

print('\n✓ Phase 2 demonstration complete.')
print('Next: Phase 3 — EvaluationCommission + EvolutionEngine (crossover + Elo selection)')

21:40:07  INFO      mre.agents.agent  Agent 7d295264 solving: Solve x**2 - 4 = 0
21:40:07  INFO      mre.agents.agent  Agent 7d295264 finished: solved=True, answer=Solution for [x]: [-2, 2]
21:40:07  INFO      mre.agents.agent  Agent 7d295264 solving: Solve 2*x - 10 = 0
21:40:07  INFO      mre.agents.agent  Agent 7d295264 finished: solved=True, answer=Solution for [x]: [5]
21:40:07  INFO      mre.agents.agent  Agent 7d295264 solving: Solve x**2 + x - 6 = 0
21:40:07  INFO      mre.agents.agent  Agent 7d295264 finished: solved=True, answer=Solution for [x]: [-3, 2]
21:40:07  INFO      mre.agents.agent  Agent 7d295264 solving: Solve x - 3 = 0
21:40:07  INFO      mre.agents.agent  Agent 7d295264 finished: solved=True, answer=Solution for [x]: [3]
Parent fitness  : 0.9900
21:40:07  INFO      mre.agents.agent  Agent b42e32a6 solving: Solve x**2 - 4 = 0
21:40:07  INFO      mre.agents.agent  Agent b42e32a6 finished: solved=True, answer=Solution for [x]: [-2, 2]
21:40:07  INFO      mre.agents.a